In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable
groq_api_key=os.getenv("GROQ_API_KEY")

## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="Data Analyst AGENT"


In [3]:
file_path = r"sales_data_sample.csv"

df = pd.read_csv(file_path, encoding='latin1')

df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [4]:
df.columns

Index(['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER',
       'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID',
       'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE',
       'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',
       'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME',
       'DEALSIZE'],
      dtype='str')

In [5]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001C1205DCEF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C11FC5BF80>, model_name='llama-3.3-70b-versatile', groq_api_key=SecretStr('**********'))

## Agent

In [6]:
from langchain_experimental.agents import create_pandas_dataframe_agent


In [7]:
Custom_Prefix="""
When you asked to give an answer please ensure to answer in such a way that it can be easily read and understood by a human.
Also try to consume lesser Tokens as much as possible. If you are asked to give an answer in a tabular format, please ensure to give the answer in a tabular format only.

"""

In [9]:
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=Custom_Prefix,
    verbose=False,
    allow_dangerous_code=True
)

In [ ]:
response = agent.invoke(""" Give me the overall performance of the Dataset and show me in tabular format for below items.
             top 3 customers with sales amount
             top 3 product lines with sales amount
             top 3 cities with sales amount""")

print(response)

{'input': ' Give me the overall performance of the Dataset and show me in tabular format for below items.\n\n             top 3 cities with sales amount', 'output': '| CITY          |   SALES |\n|:---------------|--------:|\n| San Francisco |  5205.27 |\n| Paris          |  3884.34 |\n| Pasadena       |  3746.7  |'}
